In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
from verimon.utils import setup_logging
setup_logging()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
from verimon import loaders
from verimon.unrolling import simulator_unroll
from stormvogel.mapping import stormvogel_to_stormpy

mc_sl_2x2 = "../tests/snake_ladder/mc_2x2.pm"
mon_sl_2x2 = "../tests/snake_ladder/mon_2x2.pm"

mc_sl_2x2_deadlock = "../tests/snake_ladder/mc_2x2_deadlock.pm"
mon_sl_2x2_deadlock = "../tests/snake_ladder/mon_2x2_deadlock.pm"

mc_sl_nxn = "../tests/snake_ladder/mc_nxn.pm"
mon_sl_nxn = "../tests/snake_ladder/mon_nxn.pm"

mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"

mc_simple = "../tests/simple/mc1.pm"
mon_simple = "../tests/simple/mon1.pm"

# mc, n, ladders, snakes = loaders.load_defined_snl(mc_sl_nxn)
mon_sv = loaders.load_dfa(mon_sl_nxn)
mon_sv = simulator_unroll(mon_sv, 5)
mon = stormvogel_to_stormpy(mon_sv)

In [5]:
n, ladders, snakes = loaders.random_snl_board(5**2)

# Random snakes and ladders
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
# mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)

In [7]:
from stormvogel.mapping import stormpy_to_stormvogel
from stormvogel.show import show

mc_sv = stormpy_to_stormvogel(mc)
loaders._add_valuation_to_sv_labels(mc, mc_sv)
show(mc_sv)
show(mon_sv)
print(mon)

Output()

Output()

Output()

Output()

-------------------------------------------------------------- 
Model type: 	MDP (sparse)
States: 	5
Transitions: 	18
Choices: 	18
Reward Models:  none
State Labels: 	10 labels
   * init -> 1 item(s)
   * horizon -> 1 item(s)
   * step=3 -> 1 item(s)
   * [l=0] -> 1 item(s)
   * step=0 -> 1 item(s)
   * step=1 -> 1 item(s)
   * step=2 -> 1 item(s)
   * step=4 -> 1 item(s)
   * accepting -> 4 item(s)
   * [l=1] -> 4 item(s)
Choice Labels: 	3 labels
   * normal -> 6 item(s)
   * snake -> 6 item(s)
   * ladder -> 6 item(s)
-------------------------------------------------------------- 



In [8]:
from verimon.generator import Verifier
from time import time

t = time()
pomdp = Verifier(mc, mon, "good")
# pomdp.apply_spec('P>=0.5 [ F<=3 "good"]')
pomdp.create_product()
print(f"--------------------\nStarting Paynt after {time() - t}s")
assignment = pomdp.check_paynt_prop('Pmax=? [ (F "goal") ]')

--------------------
Starting Paynt after 0.0028018951416015625s
INFO:2024-10-24 17:01:36,813 - (0.00s) - generator.py - --------------------
Synthesis summary:
optimality objective: Pmax=? [F "goal"] 

method: AR, synthesis time: 0.03 s
number of holes: 5, family size: 768, quotient: 62 states / 241 actions
explored: 100 %
MDP stats: avg MDP size: 44, iterations: 13

feasible: yes
--------------------
 


In [9]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

induced_mc, poss = pomdp.simulate_paynt_assignment(assignment, 100)

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in pomdp.mc.states.values() 
                if "good" in state.labels]
player_path = poss

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

s2, labels=init


AttributeError: 'stormpy.storage.storage.SparsePomdp' object has no attribute 'transitions'

In [9]:
from stormvogel.show import show
from stormvogel.mapping import stormpy_to_stormvogel
from stormpy import model_checking, parse_properties

imc =  stormpy_to_stormvogel(induced_mc)
result_goal = model_checking(induced_mc, parse_properties('Pmax=? [F "goal"]')[0])
result_stop = model_checking(induced_mc, parse_properties('Pmax=? [F "stop"]')[0])
prop_goal = result_goal.at(induced_mc.initial_states[0])
prop_stop = result_stop.at(induced_mc.initial_states[0])
print(f"probability to reach goal={prop_goal} and stop={prop_stop}. Total={prop_goal+prop_stop}")
show(imc)


probability to reach goal=0.9759216081772576 and stop=0.0240783918227404. Total=0.999999999999998


Output()

Output()

In [ ]:
# scheduler = pomdp.check_storm_prop('Pmax=? [ F "goal" ]')